# Stroke-5 Format and Proper Batching

Last week we used stroke-3: `(dx, dy, pen_lifted)`. This week we upgrade to **stroke-5**: `(dx, dy, p1, p2, p3)` -- the format used in the SketchRNN paper and in our full hierarchical model.

We also fix our DataLoader to handle variable-length sequences properly using `pack_padded_sequence`.

## 1. From stroke-3 to stroke-5

In stroke-3, `pen_lifted` was a single binary flag. In stroke-5, the pen state is split into three mutually exclusive flags:

- `p1 = 1` : pen is on the paper (drawing)
- `p2 = 1` : pen is lifted (end of this stroke, more strokes follow)
- `p3 = 1` : drawing is complete (end of sequence)

Only one of p1, p2, p3 is 1 at any step. This is a one-hot encoding of the pen state.

In [ ]:
import numpy as np
import json
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

def drawing_to_stroke5(drawing, max_len=200):
    '''
    Convert a Quick Draw drawing to stroke-5 format.
    Output shape: (seq_len + 1, 5)
    Columns: [dx, dy, p1, p2, p3]
    A special end-of-sequence token [0, 0, 0, 0, 1] is appended at the end.
    '''
    strokes = []
    for stroke in drawing:
        xs, ys = stroke[0], stroke[1]
        for i in range(len(xs)):
            dx = xs[i] - xs[i-1] if i > 0 else 0
            dy = ys[i] - ys[i-1] if i > 0 else 0
            if i < len(xs) - 1:
                p1, p2, p3 = 1, 0, 0   # pen on paper
            else:
                p1, p2, p3 = 0, 1, 0   # pen lifted
            strokes.append([dx, dy, p1, p2, p3])

    # Append end-of-sequence token
    strokes.append([0, 0, 0, 0, 1])

    s5 = np.array(strokes, dtype=np.float32)
    if len(s5) > max_len:
        s5 = s5[:max_len]
        s5[-1] = [0, 0, 0, 0, 1]   # force last step to be EOS
    return s5

# Test
import urllib.request, os
os.makedirs('data', exist_ok=True)
path = 'data/cat.ndjson'
if not os.path.exists(path):
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/quickdraw_dataset/full/simplified/cat.ndjson', path)

drawings = []
with open(path) as f:
    for line in f:
        drawings.append(json.loads(line))
        if len(drawings) >= 10: break

s5 = drawing_to_stroke5(drawings[0]['drawing'])
print('Stroke-5 shape:', s5.shape)
print('First 5 rows (dx, dy, p1, p2, p3):')
print(s5[:5])
print('Last row (EOS token):', s5[-1])


In [ ]:
# TODO: Check that at every step exactly one of p1, p2, p3 is 1.
#       Hint: s5[:, 2:].sum(axis=1) should be all ones.
# YOUR CODE HERE


In [ ]:
# TODO: Convert drawings[2] to stroke-5 and print how many steps have p2=1 (pen lifts).
#       What does this number correspond to visually?
# YOUR CODE HERE


## 2. Normalisation for stroke-5

Same as before -- normalise dx and dy, leave the pen state columns unchanged.

In [ ]:
def normalise_stroke5(stroke5):
    s = stroke5.copy()
    coords = s[:, :2]
    s[:, :2] = (coords - coords.mean(axis=0)) / (coords.std(axis=0) + 1e-8)
    return s

s5_norm = normalise_stroke5(s5)
print('dx/dy mean after normalisation:', s5_norm[:, :2].mean(axis=0).round(3))
print('dx/dy std  after normalisation:', s5_norm[:, :2].std(axis=0).round(3))
print('Pen states unchanged:', s5_norm[:, 2:].sum(axis=1)[:5])   # should all be 1


## 3. Updated Dataset

In [ ]:
class QuickDrawDataset(Dataset):
    def __init__(self, path, max_len=200, max_samples=5000):
        self.samples = []
        with open(path) as f:
            for i, line in enumerate(f):
                if i >= max_samples: break
                d  = json.loads(line)
                s5 = drawing_to_stroke5(d['drawing'], max_len=max_len)
                s5 = normalise_stroke5(s5)
                self.samples.append(torch.tensor(s5, dtype=torch.float32))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

def collate_fn(batch):
    lengths = [seq.shape[0] for seq in batch]
    padded  = pad_sequence(batch, batch_first=True, padding_value=0.0)
    return padded, lengths

dataset = QuickDrawDataset(path, max_len=200, max_samples=3000)
loader  = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

padded, lengths = next(iter(loader))
print('Batch shape:', padded.shape)   # (32, max_len_in_batch, 5)
print('Min length:', min(lengths), '  Max length:', max(lengths))


In [ ]:
# TODO: Why does padding_value=0.0 work for stroke-5 but is slightly ambiguous?
#       Hint: what does [0, 0, 0, 0, 0] mean as a stroke? Is it a valid stroke-5 vector?
#       How could you make padding unambiguous?
# YOUR CODE HERE


## Done!

Stroke-5 format is ready. All subsequent notebooks use this format.
Next: `conductor_worker.ipynb`